# 06 — SVM (Support Vector Machine)
**Zomato Project · Phase 6**

**Pipeline position:**
```
01_Profiling → 02_Cleaning → 03_Feature_Engineering → 04_DecisionTree → 05_LightGBM → [06_SVM] → 07_Model_Comparison
```

**Consumes outputs of:** `03_Feature_Engineering_updated.ipynb`

Datasets received (SVM‑ready, scaled):
- `X_train_svm_reg_scaled.csv` / `X_test_svm_reg_scaled.csv` — regression feature set (scaled)
- `X_train_svm_clf_scaled.csv` / `X_test_svm_clf_scaled.csv` — classification feature set (scaled)
- `y_train_svm_reg.csv` / `y_test_svm_reg.csv` — regression targets
- `y_train_svm_clf.csv` / `y_test_svm_clf.csv` — classification targets
- `svm_reg_scaler.joblib` / `svm_clf_scaler.joblib` — fitted StandardScalers

All cleaning, imputation, encoding, feature creation, and scaling are complete. **This notebook performs only model development.**

**Objective:**  
To evaluate a **margin‑based learning paradigm** (SVM) on the same engineered features. SVM constructs decision boundaries that maximise the margin between classes, offering a fundamentally different approach compared to tree‑based ensembles. The primary metric for classification is **Matthews Correlation Coefficient (MCC)** because it is robust to class imbalance.

**Important note on target leakage:**  
The engineered feature `RPI` (Restaurant Popularity Index) is derived from the rating `rate` and therefore introduces **target leakage**. We will drop `RPI` from the feature set before training any SVM model to ensure valid performance estimates.

**Reproducibility:** `random_state=42` is used throughout. Running this notebook on the same Feature Engineering outputs always produces identical results.

## 1 · Why SVM?

| Property | Practical Benefit |
|----------|-------------------|
| **Maximum‑margin separation** | Finds the optimal hyperplane that maximises the margin between classes – theoretically well‑grounded |
| **Kernel trick** | Can model complex non‑linear relationships without explicitly transforming features (via RBF, poly) |
| **Regularisation control** | The `C` parameter directly controls the trade‑off between margin width and misclassification, providing built‑in regularisation |
| **Support vectors** | The decision boundary depends only on a subset of training points (support vectors), making it memory‑efficient at prediction time |

**Compared to previous models:**  
- Decision Tree – interpretable but prone to overfitting  
- LightGBM – ensemble boosting, high performance but complex  
- SVM – geometrical approach; different bias‑variance trade‑off, particularly interesting for classification (MCC).

## 2 · Imports & Configuration

In [1]:
import pandas as pd
import numpy as np
import warnings
import joblib
import time
import json
from pathlib import Path
from datetime import datetime

# Sklearn – SVM
from sklearn.svm import SVR, SVC
from sklearn.preprocessing import StandardScaler

# Sklearn – metrics
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, classification_report
)

# Sklearn – tuning & validation
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_validate, learning_curve

# Plotting
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)

RANDOM_STATE = 42
DATA_DIR     = Path('/Users/huntstar/Projects/Zomato_project/Data/')
MODEL_DIR    = Path('/Users/huntstar/Projects/Zomato_project/Models/')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RATING_LABELS = ['Poor', 'Average', 'Good', 'Excellent']
RATING_MAP    = {0: 'Poor', 1: 'Average', 2: 'Good', 3: 'Excellent'}

print('Libraries loaded.')
print(f'Data dir  → {DATA_DIR}')
print(f'Model dir → {MODEL_DIR}')
RUN_FULL_ANALYSIS = True
SENSITIVITY_SAMPLE_SIZE = 5000
SEARCH_SAMPLE_SIZE = 10000

Libraries loaded.
Data dir  → /Users/huntstar/Projects/Zomato_project/Data
Model dir → /Users/huntstar/Projects/Zomato_project/Models


## 3 · Load SVM‑Ready Datasets

We load the **scaled** datasets prepared in the Feature Engineering notebook. We also load the fitted `StandardScaler` for later deployment.

In [2]:
# Regression
X_train_reg = pd.read_csv(DATA_DIR / 'X_train_svm_reg_scaled.csv')
X_test_reg  = pd.read_csv(DATA_DIR / 'X_test_svm_reg_scaled.csv')
scaler_reg  = joblib.load(DATA_DIR / 'svm_reg_scaler.joblib')

y_train_reg = pd.read_csv(DATA_DIR / 'y_train_svm_reg.csv').squeeze()
y_test_reg  = pd.read_csv(DATA_DIR / 'y_test_svm_reg.csv').squeeze()

# Classification
X_train_clf = pd.read_csv(DATA_DIR / 'X_train_svm_clf_scaled.csv')
X_test_clf  = pd.read_csv(DATA_DIR / 'X_test_svm_clf_scaled.csv')
scaler_clf  = joblib.load(DATA_DIR / 'svm_clf_scaler.joblib')

y_train_clf = pd.read_csv(DATA_DIR / 'y_train_svm_clf.csv').squeeze().astype(int)
y_test_clf  = pd.read_csv(DATA_DIR / 'y_test_svm_clf.csv').squeeze().astype(int)


# ── Remove RPI (target leakage) if present ──────────────────────────────────
# 03_Feature_Engineering already excludes rpi from all SVM datasets.
# This guard handles the edge case if a stale file is loaded.
RPI_COL = 'rpi'
for _name, _df in [('X_train_reg', X_train_reg), ('X_test_reg', X_test_reg),
                   ('X_train_clf', X_train_clf), ('X_test_clf', X_test_clf)]:
    if RPI_COL in _df.columns:
        print(f'⚠️  Removing "{RPI_COL}" from {_name} – target leakage detected.')
        _df.drop(columns=[RPI_COL], inplace=True)
print('✓ RPI check complete – safe to proceed.')

print('\nDataset shapes:')
print(f'  X_train_reg  : {X_train_reg.shape}')
print(f'  X_test_reg   : {X_test_reg.shape}')
print(f'  X_train_clf  : {X_train_clf.shape}')
print(f'  X_test_clf   : {X_test_clf.shape}')
print(f'  y_train_reg  : {y_train_reg.shape}  | range [{y_train_reg.min():.1f}, {y_train_reg.max():.1f}]')
print(f'  y_test_reg   : {y_test_reg.shape}')
print(f'  y_train_clf  : {y_train_clf.shape}  | classes {sorted(y_train_clf.unique())}')
print(f'  y_test_clf   : {y_test_clf.shape}')

✓ RPI check complete – safe to proceed.

Dataset shapes:
  X_train_reg  : (33332, 224)
  X_test_reg   : (8333, 224)
  X_train_clf  : (33332, 224)
  X_test_clf   : (8333, 224)
  y_train_reg  : (33332,)  | range [1.8, 4.9]
  y_test_reg   : (8333,)
  y_train_clf  : (33332,)  | classes [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
  y_test_clf   : (8333,)


## 4 · Dataset Validation

Ensure that the loaded data is consistent and correctly pre‑processed.

In [3]:
print('=== Dataset Validation ===')

assert len(X_train_reg) == len(y_train_reg), 'Reg train size mismatch'
assert len(X_test_reg)  == len(y_test_reg),  'Reg test size mismatch'
assert len(X_train_clf) == len(y_train_clf), 'Clf train size mismatch'
assert len(X_test_clf)  == len(y_test_clf),  'Clf test size mismatch'

assert X_train_reg.isnull().sum().sum() == 0, 'X_train_reg has nulls'
assert X_test_reg.isnull().sum().sum()  == 0, 'X_test_reg has nulls'
assert X_train_clf.isnull().sum().sum() == 0, 'X_train_clf has nulls'
assert X_test_clf.isnull().sum().sum()  == 0, 'X_test_clf has nulls'

assert (X_train_reg.columns == X_test_reg.columns).all(), 'Reg feature names mismatch'
assert (X_train_clf.columns == X_test_clf.columns).all(), 'Clf feature names mismatch'

# Check scaling: mean ≈ 0, std ≈ 1 for train (approx)
means = X_train_reg.mean(numeric_only=True)
stds = X_train_reg.std(ddof=0, numeric_only=True)

assert np.all(np.abs(means) < 1e-2), \
    "Means are not approximately zero"

non_constant = stds[stds > 0]

assert np.all((non_constant > 0.95) & (non_constant < 1.05)), \
    "Scaled non-constant features are not approximately standardised"

print(f"✓ {len(non_constant)} non-constant features correctly standardised")

constant_cols = stds[stds == 0]

if len(constant_cols):
    print(f"\nConstant columns ({len(constant_cols)}):")
    for col in constant_cols.index:
        print("  •", col)

# Classification distribution
print('\nClassification target distribution (train):')
dist = y_train_clf.value_counts().sort_index()
for k, v in dist.items():
    print(f'  {RATING_MAP[k]:<10} ({k}): {v:,}  ({v/len(y_train_clf)*100:.1f}%)')

print(f'\nReg features   : {X_train_reg.shape[1]}')
print(f'Clf features   : {X_train_clf.shape[1]}')
print(f'Training samples (reg): {len(X_train_reg):,}')
print(f'Training samples (clf): {len(X_train_clf):,}')
print(f'Testing samples  (reg): {len(X_test_reg):,}')
print(f'Testing samples  (clf): {len(X_test_clf):,}')

=== Dataset Validation ===
✓ 216 non-constant features correctly standardised

Constant columns (8):
  • location_Jakkur
  • location_Unknown
  • rest_type_Bakery, Kiosk
  • rest_type_Bakery, Sweet Shop
  • rest_type_Bar, Cafe
  • rest_type_Mess, Quick Bites
  • rest_type_Pop Up
  • rest_type_Sweet Shop, Dessert Parlor

Classification target distribution (train):
  Poor       (0): 230  (0.7%)
  Average    (1): 11,197  (33.6%)
  Good       (2): 18,638  (55.9%)
  Excellent  (3): 3,267  (9.8%)

Reg features   : 224
Clf features   : 224
Training samples (reg): 33,332
Training samples (clf): 33,332
Testing samples  (reg): 8,333
Testing samples  (clf): 8,333


## 5 · Why Scaling is Mandatory for SVM

SVM constructs decision boundaries by maximising the margin between classes in the feature space. The margin is measured in Euclidean distance from the decision hyperplane. If one feature has a much larger scale than another (e.g., `votes` up to 16,832 vs `cuisine_count` up to 35), the optimisation will be dominated by the larger feature, and the smaller features will have little influence. Standardisation ensures that each feature contributes equally to the margin calculation, making SVM both effective and stable.

This is why scaling was performed in the Feature Engineering notebook (using `StandardScaler`), and we load the scaled data here. All features now have mean ≈ 0 and standard deviation ≈ 1.

## 6 · Helper Functions

Same metrics and plotting utilities as previous notebooks, with minor additions for SVM-specific outputs.

In [4]:
def regression_metrics(y_true, y_pred, label=''):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    if label:
        print(f'  [{label}]')
    print(f'    RMSE : {rmse:.4f}')
    print(f'    MAE  : {mae:.4f}')
    print(f'    R²   : {r2:.4f}')
    return {'rmse': rmse, 'mae': mae, 'r2': r2}

def classification_metrics(y_true, y_pred, label=''):
    acc      = accuracy_score(y_true, y_pred)
    rec_mac  = recall_score(y_true, y_pred, average='macro', zero_division=0)
    rec_wt   = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1_wt    = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    mcc      = matthews_corrcoef(y_true, y_pred)
    if label:
        print(f'  [{label}]')
    print(f'    Accuracy          : {acc:.4f}')
    print(f'    Recall (Macro)    : {rec_mac:.4f}  (secondary)')
    print(f'    Recall (Weighted) : {rec_wt:.4f}')
    print(f'    F1 (Weighted)     : {f1_wt:.4f}')
    print(f'    MCC               : {mcc:.4f}  ← primary metric')
    return {'accuracy': acc, 'recall_macro': rec_mac, 'recall_weighted': rec_wt,
            'f1_weighted': f1_wt, 'mcc': mcc}

def save_figure(fig, filename):
    path = MODEL_DIR / filename
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  Figure saved → {path.name}')

print('Helper functions defined.')

Helper functions defined.


## 7 · Baseline SVM Models (Default Parameters)

In [5]:
# ── Baseline SVR ──────────────────────────────────────────────────────────
baseline_svr = SVR(kernel='rbf', C=1.0, gamma='scale')
baseline_svr.fit(X_train_reg, y_train_reg)
y_pred_svr_train = baseline_svr.predict(X_train_reg)
y_pred_svr_test  = baseline_svr.predict(X_test_reg)

print('BASELINE SVR (default params)')
print(f'  kernel : {baseline_svr.kernel}')
print(f'  C      : {baseline_svr.C}')
print(f'  gamma  : {baseline_svr.gamma}')
print()
reg_train_metrics_base = regression_metrics(y_train_reg, y_pred_svr_train, 'Train')
print()
reg_test_metrics_base  = regression_metrics(y_test_reg,  y_pred_svr_test,  'Test')

# ── Baseline SVC ──────────────────────────────────────────────────────────
baseline_svc = SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced')
baseline_svc.fit(X_train_clf, y_train_clf)
y_pred_svc_train = baseline_svc.predict(X_train_clf)
y_pred_svc_test  = baseline_svc.predict(X_test_clf)

print('\nBASELINE SVC (default params + class_weight="balanced")')
print(f'  kernel : {baseline_svc.kernel}')
print(f'  C      : {baseline_svc.C}')
print(f'  gamma  : {baseline_svc.gamma}')
print(f'  Support vectors : {len(baseline_svr.support_):,}')
print(f'  class_weight : {baseline_svc.class_weight}')
print(f'  Support vectors : {len(baseline_svc.support_):,}')
print()
clf_train_metrics_base = classification_metrics(y_train_clf, y_pred_svc_train, 'Train')
print()
clf_test_metrics_base  = classification_metrics(y_test_clf,  y_pred_svc_test,  'Test')

BASELINE SVR (default params)
  kernel : rbf
  C      : 1.0
  gamma  : scale

  [Train]
    RMSE : 0.2758
    MAE  : 0.1812
    R²   : 0.6083

  [Test]
    RMSE : 0.2878
    MAE  : 0.1937
    R²   : 0.5719

BASELINE SVC (default params + class_weight="balanced")
  kernel : rbf
  C      : 1.0
  gamma  : scale
  Support vectors : 19,932
  class_weight : balanced
  Support vectors : 25,889

  [Train]
    Accuracy          : 0.6766
    Recall (Macro)    : 0.8196  (secondary)
    Recall (Weighted) : 0.6766
    F1 (Weighted)     : 0.6885
    MCC               : 0.5289  ← primary metric

  [Test]
    Accuracy          : 0.6570
    Recall (Macro)    : 0.7545  (secondary)
    Recall (Weighted) : 0.6570
    F1 (Weighted)     : 0.6681
    MCC               : 0.4992  ← primary metric


## 8 · Hyperparameter Sensitivity Analysis

Instead of tree depth, we vary SVM hyperparameters one at a time to understand their impact.

In [6]:
# ── Small datasets for sensitivity analysis only ──────────────────────────────
# These are NOT used anywhere else. Full datasets are preserved for Section 9+.
from sklearn.model_selection import train_test_split

X_train_reg_small, _, y_train_reg_small, _ = train_test_split(
    X_train_reg, y_train_reg,
    train_size=SENSITIVITY_SAMPLE_SIZE,
    random_state=42
)

X_train_clf_small, _, y_train_clf_small, _ = train_test_split(
    X_train_clf, y_train_clf,
    train_size=SENSITIVITY_SAMPLE_SIZE,
    stratify=y_train_clf,
    random_state=42
)

print(f"Sensitivity subset sizes — Reg: {X_train_reg_small.shape}, Clf: {X_train_clf_small.shape}")

Sensitivity subset sizes — Reg: (5000, 224), Clf: (5000, 224)


In [7]:
if RUN_FULL_ANALYSIS:
    # ── Sensitivity to C (regularisation) ─────────────────────────────────────
    C_values = [0.1, 1, 10]   # 3 representative values; full tuning done by RandomizedSearchCV
    svr_c_train, svr_c_test = [], []
    svc_c_train, svc_c_test = [], []

    for C in C_values:
            print(f"Evaluating C = {C} ...")

            # SVR
            svr = SVR(kernel='rbf', C=C, gamma='scale')
            svr.fit(X_train_reg_small, y_train_reg_small)
            svr_c_train.append(np.sqrt(mean_squared_error(y_train_reg_small, svr.predict(X_train_reg_small))))
            svr_c_test.append(np.sqrt(mean_squared_error(y_test_reg,         svr.predict(X_test_reg))))

            # SVC
            svc = SVC(kernel='rbf', C=C, gamma='scale', class_weight='balanced')
            svc.fit(X_train_clf_small, y_train_clf_small)
            svc_c_train.append(recall_score(y_train_clf_small, svc.predict(X_train_clf_small), average='macro', zero_division=0))
            svc_c_test.append(recall_score(y_test_clf,         svc.predict(X_test_clf),         average='macro', zero_division=0))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(C_values, svr_c_train, 'o-', label='Train RMSE', color='steelblue')
    axes[0].plot(C_values, svr_c_test,  'o-', label='Test RMSE',  color='tomato')
    axes[0].set_xscale('log')
    axes[0].set_title('SVR – RMSE vs C')
    axes[0].set_xlabel('Regularisation Parameter (C)')          # improved label
    axes[0].set_ylabel('RMSE')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    best_idx_svr = np.argmin(svr_c_test)
    axes[0].scatter(C_values[best_idx_svr], svr_c_test[best_idx_svr],
                    marker="*", s=180, color="black", label="Best")
    axes[0].legend()

    axes[1].plot(C_values, svc_c_train, 'o-', label='Train Recall (Macro)', color='steelblue')
    axes[1].plot(C_values, svc_c_test,  'o-', label='Test Recall (Macro)',  color='tomato')
    axes[1].set_xscale('log')
    axes[1].set_title('SVC – Recall (Macro) vs C')
    axes[1].set_xlabel('Regularisation Parameter (C)')          # improved label
    axes[1].set_ylabel('Recall (Macro)')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    best_idx_svc = np.argmax(svc_c_test)
    axes[1].scatter(C_values[best_idx_svc], svc_c_test[best_idx_svc],
                    marker="*", s=180, color="black", label="Best")
    axes[1].legend()

    plt.suptitle('SVM – Sensitivity to Regularisation (C)', fontsize=14)
    plt.tight_layout()
    save_figure(fig, 'svm_sensitivity_C.png')
    plt.show()

    # Print best C values (from test performance)
    best_c_svr = C_values[np.argmin(svr_c_test)]
    best_c_svc = C_values[np.argmax(svc_c_test)]
    print(f"Best C for SVR (lowest test RMSE) : {best_c_svr}")
    print(f"Best C for SVC (highest test macro-recall) : {best_c_svc}")

    # ── Sensitivity to gamma (RBF kernel width) ──────────────────────────────
    # NOTE: These plots are for visual understanding of parameter behaviour only.
    # The production model hyperparameters are selected by RandomizedSearchCV in Section 9.
    # best_c_svr / best_c_svc come from the C sensitivity analysis above.
    # Final production hyperparameters will be determined later by RandomizedSearchCV.
    C_svr_best = best_c_svr
    C_svc_best = best_c_svc

    gamma_values = [0.001, 0.01, 0.1]   # 3 representative values; visual illustration only
    svr_g_train, svr_g_test = [], []
    svc_g_train, svc_g_test = [], []

    for gamma in gamma_values:
            print(f"Evaluating gamma = {gamma} ...")

            # SVR – using tuned C
            svr = SVR(kernel='rbf', C=C_svr_best, gamma=gamma)
            svr.fit(X_train_reg_small, y_train_reg_small)
            svr_g_train.append(np.sqrt(mean_squared_error(y_train_reg_small, svr.predict(X_train_reg_small))))
            svr_g_test.append(np.sqrt(mean_squared_error(y_test_reg,         svr.predict(X_test_reg))))

            # SVC – using tuned C
            svc = SVC(kernel='rbf', C=C_svc_best, gamma=gamma, class_weight='balanced')
            svc.fit(X_train_clf_small, y_train_clf_small)
            svc_g_train.append(recall_score(y_train_clf_small, svc.predict(X_train_clf_small), average='macro', zero_division=0))
            svc_g_test.append(recall_score(y_test_clf,         svc.predict(X_test_clf),         average='macro', zero_division=0))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(gamma_values, svr_g_train, 'o-', label='Train RMSE', color='steelblue')
    axes[0].plot(gamma_values, svr_g_test,  'o-', label='Test RMSE',  color='tomato')
    axes[0].set_xscale('log')
    axes[0].set_title('SVR – RMSE vs gamma')
    axes[0].set_xlabel('Kernel width parameter (gamma)')      # improved label
    axes[0].set_ylabel('RMSE')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(gamma_values, svc_g_train, 'o-', label='Train Recall (Macro)', color='steelblue')
    axes[1].plot(gamma_values, svc_g_test,  'o-', label='Test Recall (Macro)',  color='tomato')
    axes[1].set_xscale('log')
    axes[1].set_title('SVC – Recall (Macro) vs gamma')
    axes[1].set_xlabel('Kernel width parameter (gamma)')      # improved label
    axes[1].set_ylabel('Recall (Macro)')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.suptitle('SVM – Sensitivity to RBF kernel width (gamma)', fontsize=14)
    plt.tight_layout()
    save_figure(fig, 'svm_sensitivity_gamma.png')
    plt.show()

    # Print best gamma values (from test performance)
    best_g_svr = gamma_values[np.argmin(svr_g_test)]
    best_g_svc = gamma_values[np.argmax(svc_g_test)]
    print(f"Best gamma for SVR (lowest test RMSE) : {best_g_svr}")
    print(f"Best gamma for SVC (highest test macro-recall) : {best_g_svc}")

Evaluating C = 0.1 ...
Evaluating C = 1 ...
Evaluating C = 10 ...
  Figure saved → svm_sensitivity_C.png
Best C for SVR (lowest test RMSE) : 1
Best C for SVC (highest test macro-recall) : 1
Evaluating gamma = 0.001 ...
Evaluating gamma = 0.01 ...
Evaluating gamma = 0.1 ...
  Figure saved → svm_sensitivity_gamma.png
Best gamma for SVR (lowest test RMSE) : 0.001
Best gamma for SVC (highest test macro-recall) : 0.001


## 9 · Hyperparameter Tuning (RandomizedSearchCV)

We use **RandomizedSearchCV** with limited iterations to keep runtime reasonable. For regression we optimise negative RMSE; for classification we optimise **MCC**, the primary metric.

**Search spaces:**
- SVR: `C`, `gamma`, `epsilon` (kernel fixed to `rbf`)
- SVC: `C`, `gamma`, `class_weight` (kernel fixed to `rbf`)

Search is performed on a `SEARCH_SAMPLE_SIZE` subset for speed; best params are then used to retrain final models on the full dataset.

In [8]:
# ── Search subsets (hyperparameter tuning only) ────────────────────────────────
# Best params found here; final models are retrained on full data in Section 10.
from sklearn.model_selection import train_test_split

X_train_reg_search, _, y_train_reg_search, _ = train_test_split(
    X_train_reg, y_train_reg,
    train_size=SEARCH_SAMPLE_SIZE,
    random_state=42
)

X_train_clf_search, _, y_train_clf_search, _ = train_test_split(
    X_train_clf, y_train_clf,
    train_size=SEARCH_SAMPLE_SIZE,
    stratify=y_train_clf,
    random_state=42
)

print(f"Search subset sizes — Reg: {X_train_reg_search.shape}, Clf: {X_train_clf_search.shape}")

Search subset sizes — Reg: (10000, 224), Clf: (10000, 224)


In [9]:
if RUN_FULL_ANALYSIS:
    from sklearn.model_selection import RandomizedSearchCV
    from sklearn.metrics import make_scorer, matthews_corrcoef
    from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

    # ── Regression Search ──────────────────────────────────────────────────────
    print('Running RandomizedSearchCV for SVR (30 iterations)...')
    svr_param_dist = {
            'C': [0.1, 1, 10],
            'gamma': ['scale', 0.01, 0.1],
            'epsilon': [0.01, 0.1],
        }

    svr_search = RandomizedSearchCV(
        SVR(kernel='rbf'),
        param_distributions=svr_param_dist,
        n_iter=15,
        scoring='neg_root_mean_squared_error',
        cv=3,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbose=2
    )
    svr_search.fit(X_train_reg_search, y_train_reg_search)

    print(f'Best params (SVR) : {svr_search.best_params_}')
    print(f'Best CV RMSE      : {-svr_search.best_score_:.4f}')

    # ── Classification Search ──────────────────────────────────────────────────
    print('\nRunning RandomizedSearchCV for SVC (30 iterations)...')
    svc_param_dist = {
        'C': [0.1, 1, 10],
        'gamma': ['scale', 0.01, 0.1],
        'class_weight': [None, 'balanced']
    }

    # Use MCC as scoring metric
    mcc_scorer = make_scorer(matthews_corrcoef)

    svc_search = RandomizedSearchCV(
        SVC(kernel='rbf', random_state=RANDOM_STATE),
        param_distributions=svc_param_dist,
        n_iter=15,
        scoring=mcc_scorer,
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbose=2
    )
    svc_search.fit(X_train_clf_search, y_train_clf_search)

    print(f'Best params (SVC) : {svc_search.best_params_}')
    print(f'Best CV MCC       : {svc_search.best_score_:.4f}')

Running RandomizedSearchCV for SVR (30 iterations)...
Fitting 3 folds for each of 15 candidates, totalling 45 fits
[CV] END ......................C=0.1, epsilon=0.1, gamma=0.1; total time=   8.8s
[CV] END ...................C=0.1, epsilon=0.01, gamma=scale; total time=  10.7s
[CV] END ...................C=0.1, epsilon=0.01, gamma=scale; total time=  10.9s
[CV] END ...................C=0.1, epsilon=0.01, gamma=scale; total time=  10.9s
[CV] END ....................C=0.1, epsilon=0.01, gamma=0.01; total time=  11.0s
[CV] END ....................C=0.1, epsilon=0.01, gamma=0.01; total time=  11.0s
[CV] END ....................C=0.1, epsilon=0.01, gamma=0.01; total time=  11.1s
[CV] END .......................C=1, epsilon=0.01, gamma=0.1; total time=  11.4s
[CV] END .......................C=1, epsilon=0.01, gamma=0.1; total time=  11.7s
[CV] END .......................C=1, epsilon=0.01, gamma=0.1; total time=  11.8s
[CV] END ......................C=0.1, epsilon=0.1, gamma=0.1; total time=  

## 10 · Final Tuned Models

In [10]:
# Retrain on full dataset using the best params found on the search subset
final_svr = SVR(kernel='rbf', **{k: v for k, v in svr_search.best_params_.items()})
final_svr.fit(X_train_reg, y_train_reg)

final_svc = SVC(kernel='rbf', random_state=RANDOM_STATE, **{k: v for k, v in svc_search.best_params_.items()})
final_svc.fit(X_train_clf, y_train_clf)

y_pred_svr_train = final_svr.predict(X_train_reg)
y_pred_svr_test  = final_svr.predict(X_test_reg)

y_pred_svc_train = final_svc.predict(X_train_clf)
y_pred_svc_test  = final_svc.predict(X_test_clf)

print('FINAL SVR (tuned)')
print(f'  Params           : {svr_search.best_params_}')
print(f'  Support vectors  : {len(final_svr.support_):,} / {len(X_train_reg):,} ({(len(final_svr.support_)/len(X_train_reg))*100:.1f}%)')
print(f'  Training samples : {len(X_train_reg):,}')
print(f'  Testing samples  : {len(X_test_reg):,}')
print(f'  Features         : {X_train_reg.shape[1]}')
print(f'  Kernel : {final_svr.kernel}')

print('\nFINAL SVC (tuned)')
print(f'  Params            : {svc_search.best_params_}')
print(f'  Support vectors   : {len(final_svc.support_):,} / {len(X_train_clf):,} ({(len(final_svc.support_)/len(X_train_clf))*100:.1f}%)')
print(f'  Training samples  : {len(X_train_clf):,}')
print(f'  Testing samples   : {len(X_test_clf):,}')
print(f'  Features          : {X_train_clf.shape[1]}')
print(f'  Kernel : {final_svc.kernel}')

FINAL SVR (tuned)
  Params           : {'gamma': 'scale', 'epsilon': 0.1, 'C': 1}
  Support vectors  : 19,932 / 33,332 (59.8%)
  Training samples : 33,332
  Testing samples  : 8,333
  Features         : 224
  Kernel : rbf

FINAL SVC (tuned)
  Params            : {'gamma': 'scale', 'class_weight': None, 'C': 10}
  Support vectors   : 19,679 / 33,332 (59.0%)
  Training samples  : 33,332
  Testing samples   : 8,333
  Features          : 224
  Kernel : rbf


## 11 · Regression Evaluation

In [11]:
print('=' * 50)
print('REGRESSION EVALUATION — TUNED SVR')
print('=' * 50)
reg_train_metrics = regression_metrics(y_train_reg, y_pred_svr_train, 'Train')
print()
reg_test_metrics  = regression_metrics(y_test_reg,  y_pred_svr_test,  'Test')

overfit_gap = reg_train_metrics['rmse'] - reg_test_metrics['rmse']
print(f'\n  RMSE gap (train - test): {overfit_gap:.4f}')
if abs(overfit_gap) < 0.05:
    print('  → Model is well‑generalised (gap < 0.05)')
else:
    print('  → Some overfitting — consider more regularisation')

print(f"Generalisation Gap : {abs(overfit_gap):.4f}")

REGRESSION EVALUATION — TUNED SVR
  [Train]
    RMSE : 0.2758
    MAE  : 0.1812
    R²   : 0.6083

  [Test]
    RMSE : 0.2878
    MAE  : 0.1937
    R²   : 0.5719

  RMSE gap (train - test): -0.0120
  → Model is well‑generalised (gap < 0.05)
Generalisation Gap : 0.0120


## 12 · Residual Analysis (SVR)

In [12]:
residuals = y_test_reg.values - y_pred_svr_test

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_pred_svr_test, residuals, alpha=0.3, s=10, color='steelblue')
axes[0].axhline(0, color='red', linewidth=1.2, linestyle='--')
axes[0].set_title('Residuals vs Predicted (SVR)')
axes[0].set_xlabel('Predicted Rate')
axes[0].set_ylabel('Residual')
axes[0].grid(alpha=0.3)

axes[1].scatter(y_test_reg, y_pred_svr_test, alpha=0.3, s=10, color='steelblue')
lims = [min(y_test_reg.min(), y_pred_svr_test.min()),
        max(y_test_reg.max(), y_pred_svr_test.max())]
axes[1].plot(lims, lims, 'r--', linewidth=1.2)
axes[1].set_title('Actual vs Predicted (SVR)')
axes[1].set_xlabel('Actual Rate')
axes[1].set_ylabel('Predicted Rate')
axes[1].grid(alpha=0.3)

plt.suptitle('SVR — Residual Analysis', fontsize=13)
plt.tight_layout()
save_figure(fig, 'svm_regression_residuals.png')
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(residuals, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(0, color='red', linestyle='--', linewidth=1.5, label='Zero error')
ax.axvline(residuals.mean(), color='orange', linestyle='--', linewidth=1.2,
           label=f'Mean residual = {residuals.mean():.4f}')
ax.set_title('Residual Distribution — SVR', fontsize=13)
ax.set_xlabel('Residual')
ax.set_ylabel('Frequency')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
save_figure(fig, 'svm_residual_histogram.png')
plt.show()
print(f'Residual mean  : {residuals.mean():.4f}  (should be ~0)')
print(f'Residual std   : {residuals.std():.4f}')

  Figure saved → svm_regression_residuals.png
  Figure saved → svm_residual_histogram.png
Residual mean  : -0.0347  (should be ~0)
Residual std   : 0.2857


## 13 · Classification Evaluation (MCC primary)

In [13]:
print('=' * 50)
print('CLASSIFICATION EVALUATION — TUNED SVC')
print('=' * 50)
clf_train_metrics = classification_metrics(y_train_clf, y_pred_svc_train, 'Train')
print()
clf_test_metrics  = classification_metrics(y_test_clf,  y_pred_svc_test,  'Test')

print('\nFull Classification Report (Test):')
print(classification_report(y_test_clf, y_pred_svc_test,
                            target_names=RATING_LABELS, zero_division=0))

CLASSIFICATION EVALUATION — TUNED SVC
  [Train]
    Accuracy          : 0.8352
    Recall (Macro)    : 0.6709  (secondary)
    Recall (Weighted) : 0.8352
    F1 (Weighted)     : 0.8334
    MCC               : 0.7022  ← primary metric

  [Test]
    Accuracy          : 0.7943
    Recall (Macro)    : 0.6220  (secondary)
    Recall (Weighted) : 0.7943
    F1 (Weighted)     : 0.7920
    MCC               : 0.6283  ← primary metric

Full Classification Report (Test):
              precision    recall  f1-score   support

        Poor       0.85      0.19      0.31        58
     Average       0.77      0.74      0.76      2799
        Good       0.80      0.85      0.82      4659
   Excellent       0.84      0.71      0.77       817

    accuracy                           0.79      8333
   macro avg       0.82      0.62      0.66      8333
weighted avg       0.79      0.79      0.79      8333



## 14 · Confusion Matrix & Misclassifications

In [14]:
cm      = confusion_matrix(y_test_clf, y_pred_svc_test)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, fmt, title in [
    (axes[0], cm,      'd',    'Confusion Matrix (Counts)'),
    (axes[1], cm_norm, '.2f',  'Confusion Matrix (Normalised)'),
]:
    sns.heatmap(data, annot=True, fmt=fmt,
                xticklabels=RATING_LABELS, yticklabels=RATING_LABELS,
                cmap='Blues', ax=ax, linewidths=0.5, linecolor='white')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('SVC — Confusion Matrix', fontsize=13)
plt.tight_layout()
save_figure(fig, 'svm_confusion_matrix.png')
plt.show()

print('Top 5 Most Common Misclassifications:')
errors = [(RATING_MAP[a], RATING_MAP[p], cm[a][p])
          for a in range(4) for p in range(4) if a != p]
errors.sort(key=lambda x: -x[2])
print(f'  {"Actual":<12} → {"Predicted":<12}  Count')
print('  ' + '-' * 38)
for actual, predicted, count in errors[:5]:
    print(f'  {actual:<12} → {predicted:<12}  {count:,}')

  Figure saved → svm_confusion_matrix.png
Top 5 Most Common Misclassifications:
  Actual       → Predicted     Count
  --------------------------------------
  Average      → Good          724
  Good         → Average       594
  Excellent    → Good          233
  Good         → Excellent     107
  Poor         → Good          41


## 15 · Per‑Class Performance

In [15]:
from sklearn.metrics import precision_recall_fscore_support
prec, rec, f1, _ = precision_recall_fscore_support(y_test_clf, y_pred_svc_test, zero_division=0)

x = np.arange(len(RATING_LABELS))
width = 0.25
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - width, prec, width, label='Precision', color='steelblue')
ax.bar(x, rec, width, label='Recall', color='tomato')
ax.bar(x + width, f1, width, label='F1', color='seagreen')
ax.set_xticks(x)
ax.set_xticklabels(RATING_LABELS)
ax.set_ylabel('Score')
ax.set_title('Per‑Class Precision, Recall, and F1 — SVC')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
save_figure(fig, 'svm_per_class_scores.png')
plt.show()

print('Per‑class metrics (test set):')
for label, p, r, f in zip(RATING_LABELS, prec, rec, f1):
    print(f'  {label:<10} | Precision: {p:.3f} | Recall: {r:.3f} | F1: {f:.3f}')

  Figure saved → svm_per_class_scores.png
Per‑class metrics (test set):
  Poor       | Precision: 0.846 | Recall: 0.190 | F1: 0.310
  Average    | Precision: 0.774 | Recall: 0.740 | F1: 0.757
  Good       | Precision: 0.799 | Recall: 0.849 | F1: 0.823
  Excellent  | Precision: 0.842 | Recall: 0.709 | F1: 0.769


## 16 · Interpretation (Linear Coefficients or Support Vectors)

If the final SVC uses a linear kernel, we can interpret feature coefficients. Otherwise, for RBF kernel, we discuss support vectors.

In [16]:
if final_svc.kernel == 'linear':
    coefs = pd.Series(final_svc.coef_.flatten(), index=X_train_clf.columns)
    coefs_abs = coefs.abs().sort_values(ascending=False)
    print('Linear SVM — Top 10 coefficients (by absolute value):')
    print(coefs_abs.head(10).to_string())
    
    # Plot coefficients
    fig, ax = plt.subplots(figsize=(10, 6))
    top_coefs = coefs_abs.head(10)
    ax.barh(top_coefs.index, top_coefs.values, color='steelblue')
    ax.set_title('Linear SVC — Top 10 Feature Coefficients (absolute)')
    ax.set_xlabel('Coefficient magnitude')
    plt.tight_layout()
    save_figure(fig, 'svm_linear_coefficients.png')
    plt.show()
else:
    print(f'Kernel is {final_svc.kernel} — no direct feature coefficients. Decision boundary is defined by {len(final_svc.support_)} support vectors.')
    print('The RBF kernel maps features to a high‑dimensional space, making individual feature importance less interpretable.')
    print(f"Support Vector Ratio : {len(final_svc.support_) / len(X_train_clf):.2%}")

Kernel is rbf — no direct feature coefficients. Decision boundary is defined by 19679 support vectors.
The RBF kernel maps features to a high‑dimensional space, making individual feature importance less interpretable.
Support Vector Ratio : 59.04%


## 17 · Learning Curves

In [17]:
# ── SVR learning curve ─────────────────────────────────────────────────────
train_sizes, train_scores_lc, val_scores_lc = learning_curve(
    final_svr, X_train_reg, y_train_reg,
    train_sizes = np.linspace(0.2, 1.0, 5),
    scoring     = 'neg_root_mean_squared_error',
    cv          = 3,  # reduced for speed
    n_jobs      = -1
)
train_mean_reg = -train_scores_lc.mean(axis=1)
val_mean_reg   = -val_scores_lc.mean(axis=1)
train_std_reg  = train_scores_lc.std(axis=1)
val_std_reg    = val_scores_lc.std(axis=1)

# ── SVC learning curve ─────────────────────────────────────────────────────
train_sizes_clf, train_scores_clf, val_scores_clf = learning_curve(
    final_svc, X_train_clf, y_train_clf,
    train_sizes = np.linspace(0.2, 1.0, 5),
    scoring     = make_scorer(matthews_corrcoef),
    cv          = 3,
    n_jobs      = -1
)
train_mean_clf = train_scores_clf.mean(axis=1)
val_mean_clf   = val_scores_clf.mean(axis=1)
train_std_clf  = train_scores_clf.std(axis=1)
val_std_clf    = val_scores_clf.std(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_sizes, train_mean_reg, 'o-', color='steelblue', label='Training RMSE')
axes[0].plot(train_sizes, val_mean_reg,   'o-', color='tomato',    label='Validation RMSE')
axes[0].fill_between(train_sizes, train_mean_reg - train_std_reg, train_mean_reg + train_std_reg,
                     alpha=0.15, color='steelblue')
axes[0].fill_between(train_sizes, val_mean_reg - val_std_reg, val_mean_reg + val_std_reg,
                     alpha=0.15, color='tomato')
axes[0].set_title('Learning Curve — SVR')
axes[0].set_xlabel('Training Set Size')
axes[0].set_ylabel('RMSE')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(train_sizes_clf, train_mean_clf, 'o-', color='steelblue', label='Training MCC')
axes[1].plot(train_sizes_clf, val_mean_clf,   'o-', color='tomato',    label='Validation MCC')
axes[1].fill_between(train_sizes_clf, train_mean_clf - train_std_clf, train_mean_clf + train_std_clf,
                     alpha=0.15, color='steelblue')
axes[1].fill_between(train_sizes_clf, val_mean_clf - val_std_clf, val_mean_clf + val_std_clf,
                     alpha=0.15, color='tomato')
axes[1].set_title('Learning Curve — SVC')
axes[1].set_xlabel('Training Set Size')
axes[1].set_ylabel('MCC')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('SVM — Learning Curves', fontsize=14)
plt.tight_layout()
save_figure(fig, 'svm_learning_curve.png')
plt.show()

gap_reg = val_mean_reg[-1] - train_mean_reg[-1]
gap_clf = val_mean_clf[-1] - train_mean_clf[-1]
print(f'Regression gap: {gap_reg:.4f}  → {"Well‑generalised" if gap_reg < 0.05 else "High variance"}')
print(f'Classification gap: {gap_clf:.4f}  → {"Well‑generalised" if gap_clf < 0.05 else "High variance"}')

  Figure saved → svm_learning_curve.png
Regression gap: 0.0211  → Well‑generalised
Classification gap: -0.0947  → Well‑generalised


## 18 · Cross‑Validation Stability

In [18]:
if RUN_FULL_ANALYSIS:    
    # Regression CV
    cv_reg = cross_validate(
        final_svr, X_train_reg, y_train_reg,
        scoring = 'neg_root_mean_squared_error',
        cv      = 5,
        return_train_score = True
    )
    reg_cv_train = -cv_reg['train_score']
    reg_cv_test  = -cv_reg['test_score']

    # Classification CV
    cv_clf = cross_validate(
        final_svc, X_train_clf, y_train_clf,
        scoring = make_scorer(matthews_corrcoef),
        cv      = 5,
        return_train_score = True
    )
    clf_cv_train = cv_clf['train_score']
    clf_cv_test  = cv_clf['test_score']

    folds = list(range(1, 6))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(folds, reg_cv_test, 'o-', color='tomato', label='CV RMSE per fold')
    axes[0].axhline(reg_cv_test.mean(), color='steelblue', linestyle='--',
                    label=f'Mean = {reg_cv_test.mean():.4f}')
    axes[0].set_title('SVR — CV RMSE per Fold')
    axes[0].set_xlabel('Fold')
    axes[0].set_ylabel('RMSE')
    axes[0].set_xticks(folds)
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(folds, clf_cv_test, 'o-', color='tomato', label='CV MCC per fold')
    axes[1].axhline(clf_cv_test.mean(), color='steelblue', linestyle='--',
                    label=f'Mean = {clf_cv_test.mean():.4f}')
    axes[1].set_title('SVC — CV MCC per Fold')
    axes[1].set_xlabel('Fold')
    axes[1].set_ylabel('MCC')
    axes[1].set_xticks(folds)
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.suptitle('SVM — Cross‑Validation Stability (5 Folds)', fontsize=13)
    plt.tight_layout()
    save_figure(fig, 'svm_cv_fold_performance.png')
    plt.show()

    print('5‑Fold CV — SVR RMSE:')
    print(f'  Train: {reg_cv_train.mean():.4f} ± {reg_cv_train.std():.4f}')
    print(f'  Test : {reg_cv_test.mean():.4f} ± {reg_cv_test.std():.4f}')

    print('\n5‑Fold CV — SVC MCC:')
    print(f'  Train: {clf_cv_train.mean():.4f} ± {clf_cv_train.std():.4f}')
    print(f'  Test : {clf_cv_test.mean():.4f} ± {clf_cv_test.std():.4f}')

    print()
    print(f"Regression CV RMSE : {reg_cv_test.mean():.4f} ± {reg_cv_test.std():.4f}")
    print(f"SVC CV MCC : {clf_cv_test.mean():.4f} ± {clf_cv_test.std():.4f}")

  Figure saved → svm_cv_fold_performance.png
5‑Fold CV — SVR RMSE:
  Train: 0.2760 ± 0.0008
  Test : 0.2946 ± 0.0024

5‑Fold CV — SVC MCC:
  Train: 0.7047 ± 0.0041
  Test : 0.6225 ± 0.0078

Regression CV RMSE : 0.2946 ± 0.0024
SVC CV MCC : 0.6225 ± 0.0078


## 19 · Timing Analysis

In [19]:
# SVR timing
start = time.time()
final_svr.fit(X_train_reg, y_train_reg)
svr_train_time = time.time() - start

start = time.time()
_ = final_svr.predict(X_test_reg)
svr_pred_time = time.time() - start

# SVC timing
start = time.time()
final_svc.fit(X_train_clf, y_train_clf)
svc_train_time = time.time() - start

start = time.time()
_ = final_svc.predict(X_test_clf)
svc_pred_time = time.time() - start

print('TIMING ANALYSIS')
print(f'  {"Model":<20} {"Train Time":>12} {"Predict Time":>14}')
print('  ' + '-' * 48)
print(f'  {"SVR":<20} {svr_train_time*1000:>10.1f}ms {svr_pred_time*1000:>12.2f}ms')
print(f'  {"SVC":<20} {svc_train_time*1000:>10.1f}ms {svc_pred_time*1000:>12.2f}ms')

TIMING ANALYSIS
  Model                  Train Time   Predict Time
  ------------------------------------------------
  SVR                     60243.2ms     12421.46ms
  SVC                    104827.5ms     12387.35ms


## 20 · Comparison with Decision Tree and LightGBM

We load metrics from the previously saved metadata (Decision Tree and LightGBM) and compare with SVM.

In [20]:
# Load DT and LGBM metadata
dt_meta_path = MODEL_DIR / 'decision_tree_metadata.json'
lgb_meta_path = MODEL_DIR / 'lightgbm_metadata.json'

dt_reg_rmse = dt_reg_r2 = dt_clf_rec = dt_clf_f1 = dt_clf_mcc = None
lgb_reg_rmse = lgb_reg_r2 = lgb_clf_rec = lgb_clf_f1 = lgb_clf_mcc = None

if dt_meta_path.exists():
    with open(dt_meta_path, 'r') as f:
        dt_meta = json.load(f)
    dt_reg_rmse = dt_meta['regression']['test_rmse']
    dt_reg_r2   = dt_meta['regression']['test_r2']
    dt_clf_rec  = dt_meta['classification']['test_recall_macro']
    dt_clf_f1   = dt_meta['classification']['test_f1_weighted']
    dt_clf_mcc  = dt_meta['classification']['test_mcc']
else:
    print('DT metadata not found – using fallback values.')
    dt_reg_rmse = 0.0436; dt_reg_r2 = 0.9902; dt_clf_rec = 0.2523; dt_clf_f1 = 0.2387; dt_clf_mcc = 0.0029

if lgb_meta_path.exists():
    with open(lgb_meta_path, 'r') as f:
        lgb_meta = json.load(f)
    lgb_reg_rmse = lgb_meta['regression']['test_rmse']
    lgb_reg_r2   = lgb_meta['regression']['test_r2']
    lgb_clf_rec  = lgb_meta['classification']['test_recall_macro']
    lgb_clf_f1   = lgb_meta['classification']['test_f1_weighted']
    lgb_clf_mcc  = lgb_meta['classification']['test_mcc']
else:
    print('LightGBM metadata not found – using fallback values.')
    lgb_reg_rmse = 0.0410; lgb_reg_r2 = 0.9910; lgb_clf_rec = 0.2800; lgb_clf_f1 = 0.2500; lgb_clf_mcc = 0.0200

# SVM metrics
svm_reg_rmse = reg_test_metrics['rmse']
svm_reg_r2   = reg_test_metrics['r2']
svm_clf_rec  = clf_test_metrics['recall_macro']
svm_clf_f1   = clf_test_metrics['f1_weighted']
svm_clf_mcc  = clf_test_metrics['mcc']

print('=' * 70)
print('  COMPARISON: Decision Tree vs LightGBM vs SVM (Test Set)')
print('=' * 70)

print('\n  REGRESSION')
print(f'  {"Metric":<15} {"Decision Tree":>15} {"LightGBM":>15} {"SVM":>15}')
print('  ' + '-' * 65)
print(f'  {"RMSE":<15} {dt_reg_rmse:>15.4f} {lgb_reg_rmse:>15.4f} {svm_reg_rmse:>15.4f}')
print(f'  {"R²":<15} {dt_reg_r2:>15.4f} {lgb_reg_r2:>15.4f} {svm_reg_r2:>15.4f}')

print('\n  CLASSIFICATION')
print(f'  {"Metric":<15} {"Decision Tree":>15} {"LightGBM":>15} {"SVM":>15}')
print('  ' + '-' * 65)
print(f'  {"Recall (Macro)":<15} {dt_clf_rec:>15.4f} {lgb_clf_rec:>15.4f} {svm_clf_rec:>15.4f}')
print(f'  {"F1 (Weighted)":<15} {dt_clf_f1:>15.4f} {lgb_clf_f1:>15.4f} {svm_clf_f1:>15.4f}')
print(f'  {"MCC":<15} {dt_clf_mcc:>15.4f} {lgb_clf_mcc:>15.4f} {svm_clf_mcc:>15.4f}')
print('=' * 70)

# Determine best for each metric
best_reg_rmse = min(dt_reg_rmse, lgb_reg_rmse, svm_reg_rmse)
best_reg_r2   = max(dt_reg_r2, lgb_reg_r2, svm_reg_r2)
best_clf_rec  = max(dt_clf_rec, lgb_clf_rec, svm_clf_rec)
best_clf_f1   = max(dt_clf_f1, lgb_clf_f1, svm_clf_f1)
best_clf_mcc  = max(dt_clf_mcc, lgb_clf_mcc, svm_clf_mcc)

print('\nBest model per metric:')
print(f'  Regression RMSE : {"SVM" if svm_reg_rmse==best_reg_rmse else "LightGBM" if lgb_reg_rmse==best_reg_rmse else "Decision Tree"}')
print(f'  Regression R²   : {"SVM" if svm_reg_r2==best_reg_r2 else "LightGBM" if lgb_reg_r2==best_reg_r2 else "Decision Tree"}')
print(f'  Recall (Macro)  : {"SVM" if svm_clf_rec==best_clf_rec else "LightGBM" if lgb_clf_rec==best_clf_rec else "Decision Tree"}')
print(f'  F1 (Weighted)   : {"SVM" if svm_clf_f1==best_clf_f1 else "LightGBM" if lgb_clf_f1==best_clf_f1 else "Decision Tree"}')
print(f'  MCC             : {"SVM" if svm_clf_mcc==best_clf_mcc else "LightGBM" if lgb_clf_mcc==best_clf_mcc else "Decision Tree"}')

  COMPARISON: Decision Tree vs LightGBM vs SVM (Test Set)

  REGRESSION
  Metric            Decision Tree        LightGBM             SVM
  -----------------------------------------------------------------
  RMSE                     0.2593          0.1544          0.2878
  R²                       0.6524          0.8768          0.5719

  CLASSIFICATION
  Metric            Decision Tree        LightGBM             SVM
  -----------------------------------------------------------------
  Recall (Macro)           0.8161          0.2625          0.6220
  F1 (Weighted)            0.7667          0.3588          0.7920
  MCC                      0.6156          0.0051          0.6283

Best model per metric:
  Regression RMSE : LightGBM
  Regression R²   : LightGBM
  Recall (Macro)  : Decision Tree
  F1 (Weighted)   : SVM
  MCC             : SVM


## 21 · Export Models, Scaler, and Metadata

In [21]:
SVR_PATH        = MODEL_DIR / 'svm_regressor_v1.joblib'
SVC_PATH        = MODEL_DIR / 'svm_classifier_v1.joblib'
SCALER_REG_PATH = MODEL_DIR / 'svm_reg_scaler_v1.joblib'
SCALER_CLF_PATH = MODEL_DIR / 'svm_clf_scaler_v1.joblib'

joblib.dump(final_svr,  SVR_PATH)
joblib.dump(final_svc,  SVC_PATH)
joblib.dump(scaler_reg, SCALER_REG_PATH)
joblib.dump(scaler_clf, SCALER_CLF_PATH)

print(f'SVR saved        → {SVR_PATH.name}  ({SVR_PATH.stat().st_size / 1e3:.1f} KB)')
print(f'SVC saved        → {SVC_PATH.name}  ({SVC_PATH.stat().st_size / 1e3:.1f} KB)')
print(f'Scaler reg saved → {SCALER_REG_PATH.name}')
print(f'Scaler clf saved → {SCALER_CLF_PATH.name}')

# Reload verification
svr_loaded = joblib.load(SVR_PATH)
svc_loaded = joblib.load(SVC_PATH)
assert np.allclose(svr_loaded.predict(X_test_reg), y_pred_svr_test), 'SVR reload mismatch'
assert (svc_loaded.predict(X_test_clf) == y_pred_svc_test).all(), 'SVC reload mismatch'
print('\n✓ Reload verification passed')

# Metadata
metadata = {
    'notebook'          : '06_SVM.ipynb',
    'created_at'        : datetime.now().strftime('%Y-%m-%d %H:%M'),
    'reg_features'      : X_train_reg.columns.tolist(),
    'clf_features'      : X_train_clf.columns.tolist(),
    'n_reg_features'    : X_train_reg.shape[1],
    'n_clf_features'    : X_train_clf.shape[1],
    'train_size'        : len(X_train_reg),
    'test_size'         : len(X_test_reg),
    'rpi_removed'       : True,
    'regression': {
        'model'         : 'SVR',
        'best_params'   : svr_search.best_params_,
        'test_rmse'     : round(reg_test_metrics['rmse'], 4),
        'test_r2'       : round(reg_test_metrics['r2'], 4),
        'cv_rmse_mean'  : round(float(reg_cv_test.mean()), 4),
        'cv_rmse_std'   : round(float(reg_cv_test.std()), 4),
        'n_support_vectors' : len(final_svr.support_),
    },
    'classification': {
        'model'         : 'SVC',
        'best_params'   : svc_search.best_params_,
        'test_mcc'          : round(clf_test_metrics['mcc'], 4),
        'test_recall_macro' : round(clf_test_metrics['recall_macro'], 4),
        'test_f1_weighted'  : round(clf_test_metrics['f1_weighted'], 4),
        'cv_mcc_mean'       : round(float(clf_cv_test.mean()), 4),
        'cv_mcc_std'        : round(float(clf_cv_test.std()), 4),
        'n_support_vectors' : len(final_svc.support_),
    }
}

metadata["svr_best_params"] = svr_search.best_params_
metadata["svc_best_params"] = svc_search.best_params_

META_PATH = MODEL_DIR / 'svm_metadata.json'
with open(META_PATH, 'w') as f:
    json.dump(metadata, f, indent=2)
    
print(f'Metadata saved → {META_PATH.name}')

SVR saved        → svm_regressor_v1.joblib  (36124.5 KB)
SVC saved        → svm_classifier_v1.joblib  (36296.3 KB)
Scaler reg saved → svm_reg_scaler_v1.joblib
Scaler clf saved → svm_clf_scaler_v1.joblib

✓ Reload verification passed
Metadata saved → svm_metadata.json


## 22 · Notebook Summary

In [22]:
sep = '=' * 72
print(sep)
print('  ZOMATO PROJECT — 06_SVM SUMMARY')
print(sep)

print('\n  INPUT DATASETS')
print(f'  X_train_reg : {X_train_reg.shape}  |  X_test_reg : {X_test_reg.shape}')
print(f'  X_train_clf : {X_train_clf.shape}  |  X_test_clf : {X_test_clf.shape}')
print(f'  Reg features: {X_train_reg.shape[1]}  |  Clf features: {X_train_clf.shape[1]}')
print('  RPI removed to prevent target leakage.')

print('\n  REGRESSION RESULTS (Test Set)')
print(f'  Baseline RMSE : {reg_test_metrics_base["rmse"]:.4f}')
print(f'  Tuned RMSE    : {reg_test_metrics["rmse"]:.4f}  (Δ {reg_test_metrics["rmse"]-reg_test_metrics_base["rmse"]:+.4f})')
print(f'  Tuned R²      : {reg_test_metrics["r2"]:.4f}')
print(f'  CV RMSE       : {reg_cv_test.mean():.4f} ± {reg_cv_test.std():.4f}')

print('\n  CLASSIFICATION RESULTS (Test Set)')
print(f'  Baseline MCC : {clf_test_metrics_base["mcc"]:.4f}')
print(f'  Tuned MCC    : {clf_test_metrics["mcc"]:.4f}  (Δ {clf_test_metrics["mcc"]-clf_test_metrics_base["mcc"]:+.4f})')
print(f'  Tuned Recall (Macro) : {clf_test_metrics["recall_macro"]:.4f}')
print(f'  Tuned F1 (Weighted)  : {clf_test_metrics["f1_weighted"]:.4f}')
print(f'  CV MCC               : {clf_cv_test.mean():.4f} ± {clf_cv_test.std():.4f}')

print('\n  SUPPORT VECTORS')
print(f'  SVR : {len(final_svr.support_):,} / {len(X_train_reg):,} ({(len(final_svr.support_)/len(X_train_reg))*100:.1f}%)')
print(f'  SVC : {len(final_svc.support_):,} / {len(X_train_clf):,} ({(len(final_svc.support_)/len(X_train_clf))*100:.1f}%)')

print('\n  EXPORTED ARTIFACTS')
artifacts = [
    'svm_regressor_v1.joblib',
    'svm_classifier_v1.joblib',
    'scaler_v1.joblib',
    'svm_metadata.json',
    'svm_sensitivity_C.png',
    'svm_sensitivity_gamma.png',
    'svm_regression_residuals.png',
    'svm_residual_histogram.png',
    'svm_confusion_matrix.png',
    'svm_per_class_scores.png',
    'svm_learning_curve.png',
    'svm_cv_fold_performance.png',
]
# If linear kernel, add coefficient plot
if final_svc.kernel == 'linear':
    artifacts.append('svm_linear_coefficients.png')

for a in artifacts:
    print(f'  {a}')

print()
print("Best Regression Metric : " f"{reg_test_metrics['rmse']:.4f}")
print("Best Classification MCC : " f"{clf_test_metrics['mcc']:.4f}")

print('\n  NEXT NOTEBOOK')
print('  07_Model_Comparison.ipynb — Final comparison and model selection')
print(sep)

  ZOMATO PROJECT — 06_SVM SUMMARY

  INPUT DATASETS
  X_train_reg : (33332, 224)  |  X_test_reg : (8333, 224)
  X_train_clf : (33332, 224)  |  X_test_clf : (8333, 224)
  Reg features: 224  |  Clf features: 224
  RPI removed to prevent target leakage.

  REGRESSION RESULTS (Test Set)
  Baseline RMSE : 0.2878
  Tuned RMSE    : 0.2878  (Δ +0.0000)
  Tuned R²      : 0.5719
  CV RMSE       : 0.2946 ± 0.0024

  CLASSIFICATION RESULTS (Test Set)
  Baseline MCC : 0.4992
  Tuned MCC    : 0.6283  (Δ +0.1291)
  Tuned Recall (Macro) : 0.6220
  Tuned F1 (Weighted)  : 0.7920
  CV MCC               : 0.6225 ± 0.0078

  SUPPORT VECTORS
  SVR : 19,932 / 33,332 (59.8%)
  SVC : 19,679 / 33,332 (59.0%)

  EXPORTED ARTIFACTS
  svm_regressor_v1.joblib
  svm_classifier_v1.joblib
  scaler_v1.joblib
  svm_metadata.json
  svm_sensitivity_C.png
  svm_sensitivity_gamma.png
  svm_regression_residuals.png
  svm_residual_histogram.png
  svm_confusion_matrix.png
  svm_per_class_scores.png
  svm_learning_curve.png
  s

This notebook evaluates SVM as a maximum‑margin learner on the Zomato dataset. After dropping the leaked `RPI` feature, the tuned SVM provides a competitive baseline, particularly in classification when measured by MCC. While LightGBM still tends to dominate on structured tabular data, SVM offers a different trade‑off between interpretability (via linear coefficients if used) and prediction performance. The final comparison notebook will now decide the overall best model for this problem.